### Chapter 6.1 - Layers and Modules

A module is the basic building block PyTorch uses to organize neural network layers, parameters, and forward computations.


In [ ]:
from pathlib import Path
import tempfile
import torch


### 6.1.1 A Custom Module

#### 1. Intuition

A custom module is a class you write by inheriting from `torch.nn.Module`.

Inheritance means your class reuses behavior from a parent class. `self` is the current object. `forward` is the method that defines how inputs become outputs.

#### 2. Why this exists

Custom modules let you package parameters and computation into reusable model components.


#### 3. Examples

Define a tiny custom module.


In [ ]:
class CenteredLayer(torch.nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, X):
        return X - X.mean()

layer = CenteredLayer()
layer(torch.tensor([1.0, 2.0, 3.0]))


#### 4. Step-by-step breakdown

`class CenteredLayer(torch.nn.Module)` creates a child class of `Module`.

`super().__init__()` initializes PyTorch's module machinery.

`forward` receives input `X` and returns centered values.

Calling `layer(...)` automatically runs `forward`.

#### 5. Connection to ML systems

Research code often defines custom modules for model blocks that are not available as one built-in layer.

#### 6. Common confusion points

- `forward` is called indirectly by `layer(X)`.
- `super().__init__()` is required for PyTorch module internals.
- A module can have no trainable parameters.
- Custom modules should keep computation readable.


### 6.1.2 The Sequential Module

#### 1. Intuition

A sequential module runs several modules in order.

The output of one module becomes the input to the next module.

#### 2. Why this exists

Many networks are simple chains of layers. `Sequential` expresses that chain without writing a custom `forward` method.


#### 3. Examples

Build a small sequential network.


In [ ]:
net = torch.nn.Sequential(
    torch.nn.Linear(2, 3),
    torch.nn.ReLU(),
    torch.nn.Linear(3, 1),
)
X = torch.randn(4, 2)
net(X).shape


#### 4. Step-by-step breakdown

The first linear layer maps 2 features to 3 hidden values.

ReLU applies a nonlinear activation.

The second linear layer maps 3 hidden values to 1 output.

`Sequential` runs these modules in listed order.

#### 5. Connection to ML systems

`Sequential` is useful for straightforward feed-forward networks and simple model prototypes.

#### 6. Common confusion points

- `Sequential` is ordered.
- It is less convenient when the forward pass needs branches or loops.
- Each layer must output a shape the next layer accepts.
- It defines model structure, not training logic.


### 6.1.3 Executing Code in the Forward Propagation Method

#### 1. Intuition

The `forward` method can contain ordinary Python control flow, such as `if` statements and loops.

Control flow means code that decides what runs and how often it repeats.

#### 2. Why this exists

Some models need computations that are not a simple chain. Custom `forward` methods allow that flexibility.


#### 3. Examples

Use a loop inside `forward`.


In [ ]:
class RepeatDouble(torch.nn.Module):
    def forward(self, X):
        for _ in range(2):
            X = X * 2
        return X

module = RepeatDouble()
module(torch.tensor([1.0, 2.0]))


#### 4. Step-by-step breakdown

`forward` starts with input `X`.

The loop runs twice.

Each loop iteration doubles `X`.

The returned value is four times the original input.

#### 5. Connection to ML systems

PyTorch records the tensor operations that actually run, so dynamic forward logic can still work with automatic differentiation.

#### 6. Common confusion points

- Python control flow runs during the forward pass.
- Autograd tracks tensor operations, not comments or intentions.
- Complex `forward` methods should be documented with clear shapes.
- Debug dynamic modules with tiny inputs first.


### 6.1.4 Summary

#### 1. Intuition

Modules organize neural network computation.

A module may contain parameters, submodules, and a `forward` method.

#### 2. Why this exists

PyTorch models are module trees. Understanding modules makes model code easier to inspect and modify.


#### 3. Examples

Inspect submodules inside a sequential model.


In [ ]:
names = []
for name, module in net.named_children():
    names.append((name, type(module).__name__))
names


#### 4. Step-by-step breakdown

`named_children()` returns immediate submodules.

Each pair contains a name and module object.

`type(module).__name__` shows the module class name.

#### 5. Connection to ML systems

Large models are nested module trees; inspection tools help navigate them.

#### 6. Common confusion points

- A module can contain other modules.
- Not every module has parameters.
- Model structure is separate from optimizer state.
- Module inspection is a core debugging skill.


### 6.1.5 Exercises

#### 1. Intuition

These exercises practice creating and inspecting modules.

#### 2. Why this exists

Module fluency is required before reading larger research model definitions.


#### 3. Examples

Exercise 1: create a module that adds one.


In [ ]:
class AddOne(torch.nn.Module):
    def forward(self, X):
        return X + 1

AddOne()(torch.tensor([2.0]))


Exercise 2: inspect a sequential model's children.


In [ ]:
[(n, type(m).__name__) for n, m in net.named_children()]


#### 4. Step-by-step breakdown

Exercise 1 checks custom `forward` behavior.

Exercise 2 checks module-tree inspection.

Both are small versions of real PyTorch model work.

#### 5. Connection to ML systems

Custom modules and inspection utilities become essential when models are no longer simple chains.

#### 6. Common confusion points

- `forward` should return a tensor or structure of tensors.
- `named_children` shows immediate children, not every nested module.
- Calling a module runs `forward`.
- Keep module examples small while learning.
